In [1]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

load_dotenv

C:\Users\richi\AppData\Local\Temp\ipykernel_20948\549072077.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF")

Loaded 3 page(s) from the PDF


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s)")

Split into 19 chunk(s)


In [4]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name='notebooklm_rag'
)

print("Vector store created successfully")

Vector store created successfully


In [5]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

In [6]:
llm = ChatOpenAI(model='gpt-4o-mini')

rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,
    k=3
)

In [7]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s)")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s)

--- Document 1 ---
How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source

--- Document 2 ---
model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

--- Document 3 ---
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split i

In [8]:
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source

model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather


In [ ]:
query

'How does NotebookLM retrieve relevant information from uploaded documents?'

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

NotebookLM retrieves relevant information from uploaded documents by first parsing the document to extract its raw text content. This involves using optical character recognition (OCR) for scanned PDFs and direct text extraction for digital PDFs. The extracted text is then cleaned and normalized, and finally, it is split into overlapping chunks that preserve semantic coherence, allowing the system to reference specific chunks when generating answers.
